# Full Data Cleaning Pipeline

This notebook cleans Online Retail data, engineers features, validates output, and saves a processed dataset for downstream analytics.

In [9]:

# Imports and configuration
import sys
!{sys.executable} -m pip install pandas
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [ ]:

# Load dataset

input_path = Path('../data/raw/online_retail.csv')
output_dir = Path('../data/processed')
output_path = output_dir / 'cleaned_data.csv'

# Proper encoding for this dataset
df = pd.read_csv(input_path, encoding='ISO-8859-1')

print(f'Loaded shape: {df.shape}')

Loaded shape: (541909, 8)


In [ ]:

#  Initial exploration

print('Head of raw data:')
display(df.head())

print('Raw data info:')
df.info()

print('Missing values by column:')
display(df.isna().sum().sort_values(ascending=False))

Head of raw data:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


Raw data info:
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB
Missing values by column:


CustomerID     135080
Description      1454
InvoiceNo           0
StockCode           0
Quantity            0
InvoiceDate         0
UnitPrice           0
Country             0
dtype: int64

In [ ]:

# Data cleaning

rows_before = len(df)

# Handle missing values in CustomerID and Description
df = df.dropna(subset=['CustomerID', 'Description'])
rows_after_missing = len(df)

# Remove returns (negative quantity rows)
df = df[df['Quantity'] >= 0]
rows_after_quantity = len(df)

# Remove duplicate rows
df = df.drop_duplicates()
rows_after_dedup = len(df)

print(f'Rows before cleaning: {rows_before}')
print(f'Rows after dropping missing CustomerID/Description: {rows_after_missing}')
print(f'Rows after removing negative quantities: {rows_after_quantity}')
print(f'Rows after duplicate removal: {rows_after_dedup}')

Rows before cleaning: 541909
Rows after dropping missing CustomerID/Description: 406829
Rows after removing negative quantities: 397924
Rows after duplicate removal: 392732


In [ ]:

# Convert data types

# Convert transaction date to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

# Drop rows where conversion failed
df = df.dropna(subset=['InvoiceDate'])

# Convert CustomerID to nullable integer type
df['CustomerID'] = df['CustomerID'].astype('Int64')

# Ensure identifier and country columns are strings
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df['StockCode'] = df['StockCode'].astype(str)
df['Country'] = df['Country'].astype(str)

In [ ]:

# Feature engineering

# Revenue feature
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Date-based features
df['InvoiceMonth'] = df['InvoiceDate'].dt.month
df['InvoiceYear'] = df['InvoiceDate'].dt.year

display(df[['InvoiceDate', 'InvoiceMonth', 'InvoiceYear', 'Quantity', 'UnitPrice', 'Revenue']].head())

,InvoiceDate,InvoiceMonth,InvoiceYear,Quantity,UnitPrice,Revenue
0,2010-12-01 08:26:00,12,2010,6,2.55,15.30
1,2010-12-01 08:26:00,12,2010,6,3.39,20.34
2,2010-12-01 08:26:00,12,2010,8,2.75,22.00
3,2010-12-01 08:26:00,12,2010,6,3.39,20.34
4,2010-12-01 08:26:00,12,2010,6,3.39,20.34


In [ ]:

# Final validation

print(f'Final dataset shape: {df.shape}')
print('Final dataset info:')
df.info()

Final dataset shape: (392732, 11)
Final dataset info:
<class 'pandas.DataFrame'>
Index: 392732 entries, 0 to 541908
Data columns (total 11 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   InvoiceNo     392732 non-null  str           
 1   StockCode     392732 non-null  str           
 2   Description   392732 non-null  str           
 3   Quantity      392732 non-null  int64         
 4   InvoiceDate   392732 non-null  datetime64[us]
 5   UnitPrice     392732 non-null  float64       
 6   CustomerID    392732 non-null  Int64         
 7   Country       392732 non-null  str           
 8   Revenue       392732 non-null  float64       
 9   InvoiceMonth  392732 non-null  int32         
 10  InvoiceYear   392732 non-null  int32         
dtypes: Int64(1), datetime64[us](1), float64(2), int32(2), int64(1), str(4)
memory usage: 33.3 MB


In [ ]:

# Save cleaned dataset

output_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)

print(f'Clean dataset saved to: {output_path.resolve()}')

Clean dataset saved to: /Users/shalinisaloni/capstone-project2/data/processed/cleaned_data.csv
